# DevMentor — Interactive Demo Notebook

This notebook shows how to use DevMentor's Python API directly from a Jupyter notebook.

**Prerequisites:**
1. `pip install -e ".[notebook]"`
2. Ollama running with `mistral:7b` (or change `OLLAMA_MODEL` below)
3. `devmentor serve` running in another terminal (or use the embedded mode below)


In [ ]:
# Install devmentor if needed
# !pip install -e ..

import asyncio
from devmentor.harness.ollama import OllamaHarness
from devmentor.agent.core import DevMentorAgent, AnalysisRequest, AnalysisMode
from IPython.display import Markdown, display

In [ ]:
# Create the agent (connects to local Ollama)
harness = OllamaHarness(base_url="http://localhost:11434", model="mistral:7b")
agent = DevMentorAgent(harness)

# Check health
health = await agent.health()
print("Backend status:", health)

## Example 1 — Code Review

In [ ]:
code_with_bugs = '''
import sqlite3

def get_user(id):
    conn = sqlite3.connect("users.db")
    cursor = conn.cursor()
    cursor.execute(f"SELECT * FROM users WHERE id = {id}")
    return cursor.fetchone()
'''

request = AnalysisRequest(
    mode=AnalysisMode.REVIEW,
    code=code_with_bugs,
    language="python",
)
response = await agent.analyze(request)
display(Markdown(response.result))
print(f"\nModel: {response.model} | Tokens: {response.completion_tokens} | Latency: {response.latency_ms}ms")

## Example 2 — Debug

In [ ]:
broken_code = '''
def fibonacci(n):
    if n == 0:
        return 0
    elif n == 1:
        return 1
    else:
        return fibonacci(n-1) + fibonacci(n-2)

print(fibonacci(50))  # This hangs!
'''

request = AnalysisRequest(
    mode=AnalysisMode.DEBUG,
    code=broken_code,
    language="python",
    question="Why does fibonacci(50) hang and how do I fix it?"
)
response = await agent.analyze(request)
display(Markdown(response.result))

## Example 3 — Security Audit (JavaScript)

In [ ]:
js_code = '''
const express = require('express');
const app = express();

app.get('/user', (req, res) => {
    const userId = req.query.id;
    const query = `SELECT * FROM users WHERE id = ${userId}`;
    db.query(query, (err, results) => {
        res.json(results);
    });
});
'''

request = AnalysisRequest(
    mode=AnalysisMode.SECURITY,
    code=js_code,
    language="javascript"
)
response = await agent.analyze(request)
display(Markdown(response.result))

## Example 4 — Generate Code

In [ ]:
request = AnalysisRequest(
    mode=AnalysisMode.GENERATE,
    code="",
    language="python",
    question="a context manager that times code blocks and logs the duration"
)
response = await agent.analyze(request)
display(Markdown(response.result))

## Streaming example
See tokens arrive in real time:

In [ ]:
request = AnalysisRequest(
    mode=AnalysisMode.EXPLAIN,
    code="[x**2 for x in range(10) if x % 2 == 0]",
    language="python"
)

print("Streaming response:\n")
async for token in agent.stream_analyze(request):
    print(token, end='', flush=True)
print()